In [1]:
import tensorflow.keras as keras
import tensorflow as tf
import numpy as np
from natsort import natsorted
import datetime, os
import natsort
import math
import pickle
from matplotlib import pyplot as plt 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from keras.utils import np_utils
from keras.layers import LeakyReLU
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import LearningRateScheduler
from keras import backend as K
from matplotlib import pyplot as plt 
from matplotlib import pyplot
from tensorflow.keras.losses import MeanSquaredLogarithmicError,MeanSquaredError
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Flatten, Dense,LeakyReLU, MaxPool2D,Dropout,ReLU, AvgPool2D, Lambda, Concatenate,Conv2D, Activation,MaxPooling2D,DepthwiseConv2D,BatchNormalization
from tensorflow.keras.optimizers import SGD, RMSprop, Adam, Adadelta, Adagrad, Adamax, Nadam
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications import Xception
from tensorflow.keras.utils import plot_model
from PIL import Image

2024-04-07 05:26:38.020519: I tensorflow/core/platform/cpu_feature_guard.cc:194] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE3 SSE4.1 SSE4.2 AVX
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
cwdd='/workspace/awadh/nvidia/shreyaiitrpr/Shreya/ACPS/'
# cwdd='/ACPS/'

In [3]:
folder_path = 'depthdataset/new augmentation/w2'  # Change this to the path of your folder
w2 = sorted([os.path.join(folder_path, file) for file in os.listdir(folder_path)])
folder_path = 'depthdataset/new augmentation/w3'  # Change this to the path of your folder
w3 = sorted([os.path.join(folder_path, file) for file in os.listdir(folder_path)])
folder_path = 'depthdataset/new augmentation/w4'  # Change this to the path of your folder
w4 = sorted([os.path.join(folder_path, file) for file in os.listdir(folder_path)])
folder_path = 'depthdataset/new augmentation/w5'  # Change this to the path of your folder
w5 = sorted([os.path.join(folder_path, file) for file in os.listdir(folder_path)])
folder_path = 'depthdataset/new augmentation/w6'  # Change this to the path of your folder
w6 = sorted([os.path.join(folder_path, file) for file in os.listdir(folder_path)]) 

In [4]:
wk=['w2','w3','w4','w5','w6']
l=[w2,w3,w4,w5,w6]
w=[2,3,4,5,6]
train_x1=[]
test_x1=[]
valid_x1=[]
train_y1=[]
test_y1=[]
valid_y1=[]
i=0
for week in l:
    train_n=int(0.64*len(week))
    valid_n=int(0.16*len(week))
    test_n= len(week)-(train_n+valid_n)
    train_x1.extend(week[:train_n])
    valid_x1.extend(week[train_n:train_n+valid_n])
    test_x1.extend(week[train_n+valid_n:])
    for _ in range(train_n):
        train_y1.append(w[i])
    for _ in range(test_n):
        test_y1.append(w[i])
    for _ in range(valid_n):
        valid_y1.append(w[i])
    i=i+1 

In [5]:
import random
# Combine train_x and train_y into pairs
train_data = list(zip(train_x1, train_y1))
valid_data = list(zip(valid_x1, valid_y1))
test_data = list(zip(test_x1, test_y1))
# Shuffle the pairs
random.shuffle(train_data)
random.shuffle(valid_data)
random.shuffle(test_data)
# Unpack the shuffled pairs back into train_x and train_y
train_x, train_y = zip(*train_data)
valid_x, valid_y= zip(*valid_data)
test_x, test_y = zip(*test_data)

In [6]:
import numpy as np

def data_generator(file_names_x, file_names_y, batch_size):
    num_samples = len(file_names_x)
    num_batches = num_samples // batch_size

    while True:
        # Shuffle the indices for each epoch
        indices = np.arange(num_samples)
        np.random.shuffle(indices)

        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = (batch_idx + 1) * batch_size
            batch_indices = indices[start_idx:end_idx]

            rgb_batch_teacher = []
            depth_batch_teacher = []
            target_values_batch = []
            concatenated_img_array_batch = []
            rgb_batch_student = []
#             print(batch_indices)
            for index in batch_indices:
                file_name_x = file_names_x[index]
                file_name_y = file_names_y[index]

                # Load concatenated image
                concatenated_img_array = np.load(cwdd + str(file_name_x))

                # Split the concatenated image into RGB and depth
                rgb_img = concatenated_img_array[:, :, :3]
                depth_img = concatenated_img_array[:, :, 3:]

                # Get target value directly from the array
                target_value = file_name_y
                concatenated_img_array[:,:,:3] = concatenated_img_array[:,:,:3] / 255.0
                concatenated_img_array[:,:,3] = concatenated_img_array[:,:,3] / np.amax(concatenated_img_array[:,:,3])

                # Append data for teacher network
                concatenated_img_array_batch.append(concatenated_img_array)
                rgb_batch_teacher.append(rgb_img)
                depth_batch_teacher.append(depth_img)
                target_values_batch.append(target_value)

                # Append data for student network
                rgb_batch_student.append(rgb_img)

            yield (
                (np.array(concatenated_img_array_batch), np.expand_dims(np.array(target_values_batch), axis=1)),
                np.array(rgb_batch_student)
            )


In [7]:
batch_size=16
train_generator = data_generator(train_x,train_y,batch_size)
valid_generator = data_generator(valid_x, valid_y,batch_size)

In [8]:
import tensorflow as tf
from tensorflow.keras import layers
import tensorflow_addons as tfa
height=480
width=640
patch_size=16
num_patches=(height // patch_size) * (width // patch_size)
projection_dim=64
num_heads=4
dropout_rate=0.2
mlp_dim=256
at=[]

In [50]:
at1=[]
def create_student_model_regression():
    rgb_input = Input(shape=(height, width, 3))

    # Preprocessing layer
#     x = layers.Rescaling(scale=1.0/255)(rgb_input)
    patches = layers.Reshape((num_patches, patch_size * patch_size * 3))(layers.Lambda(lambda x: tf.image.extract_patches(x, sizes=[1, patch_size, patch_size, 1], 
                                                                strides=[1, patch_size, patch_size, 1], rates=[1, 1, 1, 1], padding='VALID'))(rgb_input))
    projection_layer = layers.Dense(units=projection_dim, activation='linear')
    x = projection_layer(patches)

    # Add position embeddings
    position_embedding_layer = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)
    positions = tf.range(start=0, limit=num_patches, delta=1)
    position_embeddings = position_embedding_layer(positions)
    x = x + position_embeddings
    attention_maps = []
    
    # Transformer Encoder layers
    for _ in range(num_heads):
        # Multi-Head Attention
        attention_output, attention_map = tfa.layers.MultiHeadAttention(num_heads=num_heads, head_size=projection_dim // num_heads, return_attn_coef=True)([x, x])
        attention_maps.append(attention_map)
        attention_output = layers.Dropout(dropout_rate)(attention_output)
        attention_output = layers.LayerNormalization(epsilon=1e-6)(attention_output + x)

        # Feed Forward Network
        ffn = layers.Dense(units=mlp_dim, activation='relu')(attention_output)
        ffn = layers.Dense(units=projection_dim, activation='linear')(ffn)
        ffn = layers.Dropout(dropout_rate)(ffn)
        x = layers.LayerNormalization(epsilon=1e-6)(ffn + attention_output)

    # Output layer
    x = layers.Flatten()(x)
    outputs = layers.Dense(units=1, activation='linear')(x)

    # Create the model
    model = tf.keras.models.Model(inputs=rgb_input, outputs=outputs)
    at1.append(attention_maps)


    return model


In [51]:
import h5py
import numpy as np
import tensorflow_addons as tfa
import tensorflow as tf
from keras.models import load_model
teacher_model = load_model('/workspace/awadh/nvidia/shreyaiitrpr/Shreya/ACPS/dump/saved wts/cafe new augmented_concat/weights_08_0.03.hdf5', compile=False)

In [52]:
from scipy.spatial.distance import pdist, squareform
def relational_loss_l1_distance(teacher,student):
    
    teacher_pair_dist1=pdist(teacher, metric='euclidean')
    student_pair_dist1=pdist(student, metric='euclidean')
    teacher_pair_dist=teacher_pair_dist1/ tf.reduce_mean(teacher_pair_dist1)
    student_pair_dist=student_pair_dist1/ tf.reduce_mean(student_pair_dist1)
    loss=tf.keras.losses.mean_squared_error(teacher_pair_dist,student_pair_dist)
    loss=tf.cast(loss, dtype=tf.float32)
#     print(loss)
    return loss

In [ ]:
# Create teacher and student models
# teacher_model = create_teacher_model_regression()
student_model = create_student_model_regression()

# Compile models
# teacher_model.compile(optimizer='adam', loss='mean_squared_error')
student_model.compile(optimizer='adam', loss='mean_squared_error')

# Define the step_decay function
def step_decay(epoch, lr0=0.001, drop=0.75, epochs_drop=1):
    temp = math.floor((1 + epoch) / epochs_drop)
    lrate = lr0 * math.pow(drop, temp)
    return lrate

# Initial learning rate
lr_schedule = step_decay(0)

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import scipy
alpha=0.30
beta=0.30
delta=0.1
l2_regularization_strength = 0.0001
gamma=0.30
num_epochs=10
num_batches=int(len(train_x)/batch_size)
num_batches_per_epoch=batch_size
num_val_batches=int(len(valid_x)/batch_size)

teacher_optimizer = tf.keras.optimizers.Adam()
student_optimizer = tf.keras.optimizers.Adam()



st_loss_train=[]
st_loss_valid=[]
prediction_train=[]
prediction_valid=[]
relational_train=[]
relational_valid=[]
layer_valid=[]
layer_train=[]
total_train=[]
total_valid=[]
l2_train=[]
l2_valid=[]


for epoch in range(num_epochs):
    lr_schedule = step_decay(epoch)
    tf.keras.backend.set_value(student_optimizer.lr, lr_schedule)
    tf.keras.backend.set_value(teacher_optimizer.lr, lr_schedule)
    
    
    student_loss_t=0
    prediction_loss_t=0
    relational_loss_t=0
    layer_loss_t=0
    total_loss_t=0
    l2_loss_t=0
    
    student_loss_v=0
    prediction_loss_v=0
    relational_loss_v=0
    layer_loss_v=0
    total_loss_v=0
    l2_loss_v=0
    
    teacherloss=0
    
    # Training
    
    
    for batch_idx in range(num_batches):
        (teacher_inputs, target_values), student_inputs = next(train_generator) 
        
        
          
        with tf.GradientTape() as tape:
            student_predictions = student_model(student_inputs, training=True)
            student_loss = sum(tf.keras.losses.mean_squared_error(target_values, student_predictions))/batch_size
            student_loss_t+=student_loss
    
            teacher_predictions_soft = teacher_model(teacher_inputs, training=False)
            
            prediction_loss = sum((tf.keras.losses.mean_squared_error(teacher_predictions_soft, student_predictions)))/batch_size
            prediction_loss_t+=prediction_loss
            
                       
            last_layer_teacher =tf.keras.models.Model(inputs=teacher_model.inputs, outputs=teacher_model.layers[-2].output)
            teacher_layer= last_layer_teacher(teacher_inputs)
            
            last_layer_student =tf.keras.models.Model(inputs=student_model.inputs, outputs=student_model.layers[-2].output)
            student_layer = last_layer_student(student_inputs)

            relational_loss = relational_loss_l1_distance(teacher_layer,student_layer)
            relational_loss_t+=relational_loss
            
            
#             print(teacher_layer.shape[1])
            b=(tf.norm(teacher_layer - student_layer, axis=1, ord='euclidean'))/np.sqrt(teacher_layer.shape[1])
            layer_loss= sum(b)/batch_size
            layer_loss_t+=layer_loss
            
            l2_loss=0
            for var in student_model.trainable_variables:
                   l2_loss+=tf.reduce_sum(tf.square(var))
                
            
            l2_loss_t+=l2_loss
            
            total_loss = (gamma*student_loss)+ (alpha * prediction_loss) + (beta * relational_loss) + (delta *layer_loss) 
#             + (l2_regularization_strength * l2_loss)
            total_loss_t+=total_loss
            if(batch_idx % 50 ==0):
                print(f"Epoch {epoch + 1}, Batch {batch_idx + 1}/{num_batches_per_epoch} - Student Loss: {student_loss}, prediction_loss loss: {prediction_loss}, relational_loss loss2: {relational_loss}, layer_loss loss 3: {layer_loss} ,Total loss: {total_loss}, L2loss:{l2_loss} ")
    


           
        student_gradients = tape.gradient(total_loss, student_model.trainable_variables)
        student_optimizer.apply_gradients(zip(student_gradients, student_model.trainable_variables))
        
       
    st_loss_train.append(student_loss)
    prediction_train.append(prediction_loss)
    relational_train.append(relational_loss)
    total_train.append(total_loss)
    layer_train.append(layer_loss)
    l2_train.append(l2_loss)
    print('***********************************************************************************************************************************************************************************************************************************************************************************************')
    print('Training')
    print(f"After Epoch {epoch + 1}, Batch {batch_idx + 1}/{num_batches_per_epoch} - Student Loss: {student_loss}, prediction_loss loss: {prediction_loss}, relational_loss loss2: {relational_loss}, layer_loss loss 3: {layer_loss} ,Total loss: {total_loss}, L2loss:{l2_loss} ")
    print('***********************************************************************************************************************************************************************************************************************************************************************************************')
    print(f"Average Epoch {epoch + 1},  Student Loss: {student_loss_t/num_batches}, prediction_loss loss: {prediction_loss_t/num_batches}, relational_loss loss2: {relational_loss_t/num_batches}, layer_loss 3: {layer_loss_t/num_batches} ,Total loss: {total_loss_t/num_batches}, L2loss:{l2_loss_t/num_batches} ")
    
    
     
    
    
   
    
    for val_batch_idx in range(num_val_batches):
        (val_teacher_inputs, val_target_values), val_student_inputs = next(valid_generator)
       
        
        
        student_predictions = student_model(val_student_inputs, training=False)
        student_loss = sum(tf.keras.losses.mean_squared_error(val_target_values, student_predictions))/batch_size
        student_loss_v+=student_loss
        
        teacher_predictions_soft = teacher_model(val_teacher_inputs, training=False)
        
        prediction_loss = sum((tf.keras.losses.mean_squared_error(teacher_predictions_soft, student_predictions)))/batch_size
        prediction_loss_v+=prediction_loss
        
        
        teacher_pred= sum(tf.keras.losses.mean_squared_error(val_target_values, teacher_predictions_soft))/batch_size
        teacherloss+=teacher_pred
        
      
                       
        last_layer_teacher =tf.keras.models.Model(inputs=teacher_model.inputs, outputs=teacher_model.layers[-2].output)
        teacher_layer= last_layer_teacher(val_teacher_inputs)

        last_layer_student =tf.keras.models.Model(inputs=student_model.inputs, outputs=student_model.layers[-2].output)
        student_layer = last_layer_student(val_student_inputs)

        relational_loss = relational_loss_l1_distance(teacher_layer,student_layer)
        relational_loss_v+=relational_loss

        
        b=(tf.norm(teacher_layer - student_layer, axis=1, ord='euclidean'))/np.sqrt(teacher_layer.shape[1])
        layer_loss= sum(b)/batch_size
        layer_loss_v+=layer_loss

#         l2_loss=0
#         for var in student_model.trainable_variables:
#                l2_loss+=tf.reduce_sum(tf.square(var))


#         l2_loss_v+=l2_loss

        total_loss = (gamma*student_loss)+ (alpha * prediction_loss) + (beta * relational_loss) + (delta *layer_loss) 
#     + (l2_regularization_strength * l2_loss)
        total_loss_v+=total_loss  
        
    st_loss_train.append(student_loss_v)
    prediction_train.append(prediction_loss_v)
    relational_train.append(relational_loss_v)
    total_train.append(total_loss_v)
    layer_train.append(layer_loss_v)
    l2_train.append(l2_loss)
    print('***********************************************************************************************************************************************************************************************************************************************************************************************')
    print('Validation')
    print(f"After Epoch {epoch + 1} Student Loss: {student_loss}, prediction_loss loss: {prediction_loss}, relational_loss loss2: {relational_loss}, layer_loss loss 3: {layer_loss} ,Total loss: {total_loss}")
    print('***********************************************************************************************************************************************************************************************************************************************************************************************')
    print(f"Average Epoch {epoch + 1}, Batch {val_batch_idx + 1}/{num_batches_per_epoch} - Student Loss: {student_loss_v/num_val_batches}, prediction_loss loss: {prediction_loss_v/num_val_batches}, relational_loss loss2: {relational_loss_v/num_val_batches}, layer_loss 3: {layer_loss_v/num_val_batches} ,Total loss: {total_loss_v/num_val_batches}, Teacher loss:{teacherloss/num_val_batches} ")
    
    
    print('\n')
    print('@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@')

    student_model.save(f'/workspace/awadh/nvidia/Shreya/ACPS/dump/saved wts/final tafe_after review_1/student_model_epoch_{epoch + 1}.h5')
    

2024-04-05 12:13:41.963958: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:648] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


Epoch 1, Batch 1/16 - Student Loss: 2.373694658279419, prediction_loss loss: 2.2286131381988525, relational_loss loss2: 0.17121948301792145, layer_loss loss 3: 1.0014092922210693 ,Total loss: 1.5321992635726929, L2loss:1854.75927734375 


2024-04-05 12:13:42.941096: I tensorflow/compiler/xla/service/service.cc:173] XLA service 0x114445f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-04-05 12:13:42.941157: I tensorflow/compiler/xla/service/service.cc:181]   StreamExecutor device (0): NVIDIA A100-SXM4-40GB, Compute Capability 8.0
2024-04-05 12:13:42.946407: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-04-05 12:13:43.113995: I tensorflow/compiler/jit/xla_compilation_cache.cc:480] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Epoch 1, Batch 51/16 - Student Loss: 2.579174757003784, prediction_loss loss: 2.3430919647216797, relational_loss loss2: 0.19422556459903717, layer_loss loss 3: 0.9908727407455444 ,Total loss: 1.6340349912643433, L2loss:1857.45166015625 
Epoch 1, Batch 101/16 - Student Loss: 2.3540146350860596, prediction_loss loss: 2.2502052783966064, relational_loss loss2: 0.13940103352069855, layer_loss loss 3: 0.9901648163795471 ,Total loss: 1.5221027135849, L2loss:1857.4686279296875 
Epoch 1, Batch 151/16 - Student Loss: 2.894325017929077, prediction_loss loss: 2.7345688343048096, relational_loss loss2: 0.2537751793861389, layer_loss loss 3: 0.9893195629119873 ,Total loss: 1.8637326955795288, L2loss:1857.4951171875 
Epoch 1, Batch 201/16 - Student Loss: 3.9825828075408936, prediction_loss loss: 3.919769763946533, relational_loss loss2: 0.18490105867385864, layer_loss loss 3: 0.9886302947998047 ,Total loss: 2.5250391960144043, L2loss:1857.6207275390625 
Epoch 1, Batch 251/16 - Student Loss: 2.02980

In [9]:
import h5py
import numpy as np
import tensorflow_addons as tfa
import tensorflow as tf
from keras.models import load_model
model = load_model('/workspace/awadh/nvidia/shreyaiitrpr/Shreya/ACPS/dump/saved wts/final tafe_after review/student_model_epoch_10.h5', compile=False)

2024-04-07 05:27:47.918543: I tensorflow/core/platform/cpu_feature_guard.cc:194] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE3 SSE4.1 SSE4.2 AVX
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-07 05:27:48.038073: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1621] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38319 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:90:00.0, compute capability: 8.0
/usr/local/lib/python3.8/dist-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer GlorotUniform is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warn

In [11]:
y_pred_all=[]
for ff in test_x:
#     print(ff)
#     if(samp%100==0):
#         print(samp)
    arr=np.load(cwdd+ str(ff))
#     print(arr.shape)
    arr[:,:,:3]=arr[:,:,:3]/255.0
    arr1=np.expand_dims(arr[:,:,:3], axis=0)
#     print(arr1.shape)
    y_pred = model.predict(arr1, verbose=0)
    y_pred_all.append(y_pred[0,0])

y_true=np.asarray(test_y)
y_pred1=np.asarray(y_pred_all)

mse_out1=tf.keras.metrics.mean_squared_error(y_true, y_pred1)
mse_out1 

2024-04-07 05:28:04.288768: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:648] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


<tf.Tensor: shape=(), dtype=float32, numpy=0.17102739>

In [12]:
y_pred_rounded = np.round(y_pred1)
y_pred_rounded

array([6., 3., 3., ..., 4., 2., 6.], dtype=float32)

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
accuracy = accuracy_score(y_true, y_pred_rounded)
accuracy

0.8613324479858344

In [14]:
conf_matrix = confusion_matrix(y_true, y_pred_rounded)
conf_matrix

array([[1260,  465,   10,    0,    0],
       [   0, 1511,  283,   21,    0],
       [   0,    3, 1694,   33,    0],
       [   0,    0,  351, 1512,    0],
       [   0,    0,    0,   87, 1806]])

In [15]:
from sklearn.metrics import classification_report
target_names = ['class 2', 'class 3', 'class 4','class 5','class 6']
print(classification_report(y_true, y_pred_rounded,target_names=target_names))

              precision    recall  f1-score   support

     class 2       1.00      0.73      0.84      1735
     class 3       0.76      0.83      0.80      1815
     class 4       0.72      0.98      0.83      1730
     class 5       0.91      0.81      0.86      1863
     class 6       1.00      0.95      0.98      1893

    accuracy                           0.86      9036
   macro avg       0.88      0.86      0.86      9036
weighted avg       0.88      0.86      0.86      9036

